# Study 870 — Industry-Leader Lead-Lag 👑

**Does the *biggest* name in a sector lead the rest?**

Hou (2007) finds that information diffuses **within an industry** from the largest firm
outward: the bellwether's return this week foreshadows its smaller peers' returns next
week (slow within-industry diffusion). Trade it — long the followers whose leader *rose*,
short those whose leader *fell*. We take the self-contained weekly version on a liquid US
cross-section (2010-01-04 → 2026-06-30, 50 names in 8 sectors).

*Numbers below are the frozen headline (`docs/results.md`); the live cells run the fast
synthetic control. Survivorship + leader designation: current-membership mega-caps,
largest-cap leaders — magnitudes are an upper bound.*


## 1. The idea in one picture

Big, closely-followed firms price sector news first; their smaller industry peers, watched by fewer eyes, catch up a beat later. So the **leader's** move this week should tip the **followers'** move next week. Buy the followers of leaders that rose, sell the followers of leaders that fell.

In [1]:
import numpy as np, pandas as pd
R = dict(spread_bps=-3.64, t_nw=-0.77, up_bps=28.79, dn_bps=35.99, gross_sharpe=-0.17, placebo_p=0.928)
print('long up-leader / short down-leader followers spread: %+.2f bps/week (NW t = %+.2f)'
      % (R['spread_bps'], R['t_nw']))
print('  followers after up-leader %+.2f bps vs after down-leader %+.2f bps'
      % (R['up_bps'], R['dn_bps']))
print('  gross weekly Sharpe (before cost): %.2f' % R['gross_sharpe'])
print('  placebo right-tail p: %.3f (nothing special in the true alignment)' % R['placebo_p'])

long up-leader / short down-leader followers spread: -3.64 bps/week (NW t = -0.77)
  followers after up-leader +28.79 bps vs after down-leader +35.99 bps
  gross weekly Sharpe (before cost): -0.17
  placebo right-tail p: 0.928 (nothing special in the true alignment)


## 2. Is the sort even wired up? A live synthetic control

We plant a leader→follower diffusion in a seeded toy world (`edge>0`) and check the detector recovers it — and that it stays *silent* on the null (`edge=0`, leaders and followers independent). No network.

In [2]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
from leader_lag import data, strategy as st
secs, lds = data.synthetic_sectors(), data.synthetic_leaders()
null = st.synthetic_detect(data.synthetic_panel(edge=0.0, seed=870, n_weeks=260), secs, lds)
planted = st.synthetic_detect(data.synthetic_panel(edge=0.6, seed=870, n_weeks=320), secs, lds)
print('null world   : spread NW t = %+.2f  (should be ~0)' % null['t_nw'])
print('planted world: spread NW t = %+.2f  (should light up)' % planted['t_nw'])

C:\Users\loutr\Dropbox\Perso\GitHub\Open-Alpha-Lab\studies\870-industry-leader-lead-lag\leader_lag\strategy.py:130: RuntimeWarning: Mean of empty slice
  foll[:, k] = np.nanmean(R[:, fi], axis=1)
C:\Users\loutr\Dropbox\Perso\GitHub\Open-Alpha-Lab\studies\870-industry-leader-lead-lag\leader_lag\strategy.py:141: RuntimeWarning: Mean of empty slice
  spread = np.nanmean(np.where(np.isfinite(contrib), contrib, np.nan), axis=1)
C:\Users\loutr\Dropbox\Perso\GitHub\Open-Alpha-Lab\studies\870-industry-leader-lead-lag\leader_lag\strategy.py:142: RuntimeWarning: Mean of empty slice
  up = np.nanmean(np.where(sign > 0, foll_next, np.nan), axis=1)
C:\Users\loutr\Dropbox\Perso\GitHub\Open-Alpha-Lab\studies\870-industry-leader-lead-lag\leader_lag\strategy.py:143: RuntimeWarning: Mean of empty slice
  dn = np.nanmean(np.where(sign < 0, foll_next, np.nan), axis=1)


null world   : spread NW t = -0.70  (should be ~0)
planted world: spread NW t = +20.79  (should light up)


C:\Users\loutr\Dropbox\Perso\GitHub\Open-Alpha-Lab\studies\870-industry-leader-lead-lag\leader_lag\strategy.py:130: RuntimeWarning: Mean of empty slice
  foll[:, k] = np.nanmean(R[:, fi], axis=1)
C:\Users\loutr\Dropbox\Perso\GitHub\Open-Alpha-Lab\studies\870-industry-leader-lead-lag\leader_lag\strategy.py:141: RuntimeWarning: Mean of empty slice
  spread = np.nanmean(np.where(np.isfinite(contrib), contrib, np.nan), axis=1)
C:\Users\loutr\Dropbox\Perso\GitHub\Open-Alpha-Lab\studies\870-industry-leader-lead-lag\leader_lag\strategy.py:142: RuntimeWarning: Mean of empty slice
  up = np.nanmean(np.where(sign > 0, foll_next, np.nan), axis=1)
C:\Users\loutr\Dropbox\Perso\GitHub\Open-Alpha-Lab\studies\870-industry-leader-lead-lag\leader_lag\strategy.py:143: RuntimeWarning: Mean of empty slice
  dn = np.nanmean(np.where(sign < 0, foll_next, np.nan), axis=1)


## 3. The honest verdict — the famous edge does *not* replicate here

On this liquid mega-cap tape the long-up-leader / short-down-leader followers spread is **-3.64 bps/week** with NW *t* = **-0.77** — statistically indistinguishable from zero, and if anything leaning the *wrong* way (followers earned a touch *more* after their leader **fell**, Welch *t* = -1.99). The permutation null actually centres *above* the observed value (right-tail p = 0.93), and the sign flips era-to-era (-14.0 bps early vs +6.0 late). The seeded synthetic control recovers a *planted* diffusion emphatically, so the sort works — there is simply no lead-lag to harvest here. Slow within-industry diffusion is a small-and-illiquid-firm effect; 50 mega-caps price sector news near-simultaneously. **Signal: None**, **Tradability: Mirage**.